# CRIS Section 1 - Overnight Summary

Six queries, comments removed. Run a cell and it returns the answer
for the parameters in its own SET block.

## v70

From A&E's working session of 24 Sep 2026.

**2.2** takes `p_premiere_only`: the first telecast of the season against the
network's 12-month premiere norm, one row with the episode and air date. This is
the season premiere answer; the planner routes it here.

**2.11** takes `p_norm_basis = 'TIME_PERIOD'`: the norm is the latest 12 months
of the network's first-run premieres airing in the same time period as the
programme, all days, position by position, with no specials, mini-series or
Curse exclusions. `p_rollup` returns the four season rows.

**2.9** takes `p_slot_min_min = 15`, the 3.1 rule: a half hour belongs to the
series' time period only if the series contributed 15 minutes to it. Matched
Jill's figures.

**2.10** steps over shorts (`p_min_leadin_min = 15`) and takes `p_rollup` for
the season view.

**2.8** returns every eligible series with `IS_TARGET`. **2.12** is ordered by
rank. **3.2** returns the whole universe with `IS_TARGET`, ties on all four
ranks in the form 7 (tied), and no entertainment-cable rank on news and
sports networks. **2.7**'s norm gets the 8p-12a window.

SQL is shown without comments. Requires `CRIS_Tables_v70`.


**Every network.** The six-network filter is gone from the fact tables, so a
question about any network returns rows rather than nothing.

**`IS_PREMIERE_CT`** is on the tables and nothing reads it yet. Cable Tracks
presence as the premiere indicator, signed off by Jill on 11 Sep 2026.
`REPEAT_IND` is untouched, so the two can be compared before anything migrates.

**1.3 network norm** now runs the latest 12 months again, ending on the most
recent Sunday with data. Season norms stay season-based.


## 1.1 Premieres vs Season Norms

**A&E case:** OV_1: How did HIST's premieres do on 5/30/25 vs their season norms?

**Expected:** SECRETS DECLASSIFIED 306 vs norm 210, +45.7%; HISTORY'S GREATEST 290 vs 279, +3.9%


In [ ]:
%%sql
SET p_network            = 'HIST';
SET p_rating_src         = 'Live+SD';
SET p_target_date        = '2025-05-30';
SET p_program            = NULL;
SET p_season             = NULL;
SET p_report_from        = NULL;
SET p_report_to          = NULL;
SET p_include_specials   = TRUE;
SET p_win_start_2959     = 2000;
SET p_win_end_2959       = 2400;
SET p_demo = NULL;
SET eff_demo = (SELECT COALESCE($p_demo, 'DEMO_AA'));
SET eff_target = (SELECT COALESCE(TO_DATE($p_target_date),
                   (SELECT MAX(LATEST_BROADCAST_DATE)
                    FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                    WHERE ($p_network IS NULL OR NETWORK_CODE = $p_network)
                      AND RATING_SOURCE = $p_rating_src)));

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH
fy AS (
    SELECT FISCAL_YEAR, FY_START, FY_END
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_FY_CALENDAR
),
target AS (
    SELECT DISTINCT t0.NETWORK_CODE, t0.PROGRAM_CODE, t0.CT_SEASON,
           IFF(REGEXP_LIKE(t0.CT_SEASON, '^(19|20)[0-9]{2}[A-Za-z]?$'),
               'FY' || fy.FISCAL_YEAR, t0.CT_SEASON)          AS NORM_KEY,
           REGEXP_LIKE(t0.CT_SEASON, '^(19|20)[0-9]{2}[A-Za-z]?$') AS IS_YEAR_LABEL,
           fy.FY_START, fy.FY_END
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT t0
    LEFT JOIN fy
      ON t0.BROADCAST_DATE BETWEEN fy.FY_START AND fy.FY_END
    WHERE ( ($p_program IS NULL
             AND t0.BROADCAST_DATE BETWEEN COALESCE(TO_DATE($p_report_from), $eff_target)
                                    AND COALESCE(TO_DATE($p_report_to),   $eff_target))
         OR ($p_program IS NOT NULL AND t0.PROGRAM_NAME ILIKE $p_program
             AND t0.BROADCAST_DATE <= $eff_target) )
      AND ($p_season IS NULL OR t0.CT_SEASON = $p_season)
      AND t0.IS_FIRST_RUN_CT
      AND ($p_rating_src IS NULL OR t0.RATING_SOURCE = $p_rating_src)
      AND ($p_network    IS NULL OR t0.NETWORK_CODE  = $p_network)
      AND ($p_include_specials OR t0.CONTENT_TYPE <> 'Special')
      AND ($p_win_start_2959 IS NULL OR t0.BCAST_START_2959 >= $p_win_start_2959)
      AND ($p_win_end_2959   IS NULL OR t0.BCAST_START_2959 <  $p_win_end_2959)
      AND t0.CT_SEASON IS NOT NULL
),
base AS (
    SELECT f.NETWORK_CODE, f.PROGRAM_CODE, f.TELECAST_NUM,
           f.PROGRAM_NAME, f.EPISODE_NAME, f.CT_SEASON,
           f.BROADCAST_DATE, f.DAY_OF_WEEK, f.START_TIME, f.BCAST_START_2959,
           IDENTIFIER($eff_demo) AS DEMO_AA, f.CONTENT_TYPE,
           f.CONTRIBUTING_DURATION AS W,
           t.NORM_KEY,
           t.IS_YEAR_LABEL
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT f
    JOIN target t
      ON  t.NETWORK_CODE = f.NETWORK_CODE
      AND t.PROGRAM_CODE = f.PROGRAM_CODE
      AND ( (NOT t.IS_YEAR_LABEL AND f.CT_SEASON = t.CT_SEASON)
         OR (t.IS_YEAR_LABEL
             AND f.BROADCAST_DATE BETWEEN t.FY_START AND t.FY_END) )
    WHERE f.IS_FIRST_RUN_CT
      AND ($p_rating_src IS NULL OR f.RATING_SOURCE = $p_rating_src)
      AND f.BROADCAST_DATE <= $eff_target
      AND ($p_include_specials OR f.CONTENT_TYPE <> 'Special')
),
prog AS (
    SELECT DISTINCT NETWORK_CODE, PROGRAM_CODE FROM target
),
hist AS (
    SELECT f.NETWORK_CODE, f.PROGRAM_CODE, f.BROADCAST_DATE,
           f.CT_SEASON,
           IFF(REGEXP_LIKE(f.CT_SEASON, '^(19|20)[0-9]{2}[A-Za-z]?$'),
               'FY' || fy.FISCAL_YEAR, f.CT_SEASON)           AS NORM_KEY,
           IDENTIFIER($eff_demo) AS DEMO_AA,
           f.CONTRIBUTING_DURATION AS W
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT f
    JOIN prog p
      ON  p.NETWORK_CODE = f.NETWORK_CODE
      AND p.PROGRAM_CODE = f.PROGRAM_CODE
    LEFT JOIN fy
      ON f.BROADCAST_DATE BETWEEN fy.FY_START AND fy.FY_END
    WHERE f.IS_FIRST_RUN_CT
      AND ($p_rating_src IS NULL OR f.RATING_SOURCE = $p_rating_src)
      AND f.BROADCAST_DATE <= $eff_target
      AND ($p_include_specials OR f.CONTENT_TYPE <> 'Special')
      AND f.CT_SEASON IS NOT NULL
),
span AS (
    SELECT NETWORK_CODE, PROGRAM_CODE, NORM_KEY,
           MIN(BROADCAST_DATE) AS S_FROM,
           MAX(BROADCAST_DATE) AS S_TO,
           SUM(DEMO_AA * W) / NULLIF(SUM(W), 0) AS S_AVG,
           COUNT(*)            AS S_TC
    FROM hist
    GROUP BY 1,2,3
),
tgt_span AS (
    SELECT NETWORK_CODE, PROGRAM_CODE, NORM_KEY,
           MIN(BROADCAST_DATE) AS SEASON_START
    FROM base
    GROUP BY 1,2,3
),
prior_pick AS (
    SELECT t.NETWORK_CODE, t.PROGRAM_CODE, t.NORM_KEY,
           s.NORM_KEY AS PRIOR_KEY,
           s.S_AVG    AS PRIOR_AVG,
           s.S_TC     AS PRIOR_TC,
           s.S_FROM   AS PRIOR_FROM,
           s.S_TO     AS PRIOR_TO
    FROM tgt_span t
    JOIN span s
      ON  s.NETWORK_CODE = t.NETWORK_CODE
      AND s.PROGRAM_CODE = t.PROGRAM_CODE
      AND s.NORM_KEY    <> t.NORM_KEY
      AND s.S_TO         < t.SEASON_START
    QUALIFY ROW_NUMBER() OVER (PARTITION BY t.NETWORK_CODE, t.PROGRAM_CODE, t.NORM_KEY
                               ORDER BY s.S_TO DESC) = 1
),
prior_prem AS (
    SELECT pp.NETWORK_CODE, pp.PROGRAM_CODE, pp.NORM_KEY,
           SUM(h.DEMO_AA * h.W) / NULLIF(SUM(h.W), 0) AS PRIOR_PREM_AVG,
           COUNT(*)       AS PRIOR_PREM_TC
    FROM prior_pick pp
    JOIN hist h
      ON  h.NETWORK_CODE   = pp.NETWORK_CODE
      AND h.PROGRAM_CODE   = pp.PROGRAM_CODE
      AND h.NORM_KEY       = pp.PRIOR_KEY
      AND h.BROADCAST_DATE = pp.PRIOR_FROM
    GROUP BY 1,2,3
),
std AS (
    SELECT b.*,
        (SUM(DEMO_AA * W) OVER (PARTITION BY b.NETWORK_CODE, b.PROGRAM_CODE, b.NORM_KEY)
         - SUM(DEMO_AA * W) OVER (PARTITION BY b.NETWORK_CODE, b.PROGRAM_CODE, b.NORM_KEY,
                                  b.BROADCAST_DATE))
        / NULLIF(
            SUM(W) OVER (PARTITION BY b.NETWORK_CODE, b.PROGRAM_CODE, b.NORM_KEY)
            - SUM(W) OVER (PARTITION BY b.NETWORK_CODE, b.PROGRAM_CODE, b.NORM_KEY,
                          b.BROADCAST_DATE), 0)                AS PROG_STD_AVG,
        COUNT(*) OVER (PARTITION BY b.NETWORK_CODE, b.PROGRAM_CODE, b.NORM_KEY)
          - COUNT(*) OVER (PARTITION BY b.NETWORK_CODE, b.PROGRAM_CODE, b.NORM_KEY,
                           b.BROADCAST_DATE)                      AS PROG_STD_TC,
        RANK() OVER (PARTITION BY b.NETWORK_CODE, b.PROGRAM_CODE, b.NORM_KEY
                     ORDER BY ROUND(b.DEMO_AA/1000,0) DESC)       AS STD_RANK,
        COUNT(*) OVER (PARTITION BY b.NETWORK_CODE, b.PROGRAM_CODE, b.NORM_KEY)
                                                                  AS STD_UNIVERSE
    FROM base b
),
joined AS (
    SELECT s.*,
           pk.PRIOR_KEY, pk.PRIOR_AVG, pk.PRIOR_TC, pk.PRIOR_FROM, pk.PRIOR_TO,
           pm.PRIOR_PREM_AVG, pm.PRIOR_PREM_TC,
           IFF(s.PROG_STD_TC > 0, s.PROG_STD_AVG, pk.PRIOR_AVG)   AS EFF_NORM,
           IFF(s.PROG_STD_TC > 0, s.PROG_STD_TC,  pk.PRIOR_TC)    AS EFF_TC,
           CASE WHEN s.PROG_STD_TC > 0 AND s.IS_YEAR_LABEL THEN 'FISCAL YEAR TO DATE'
                WHEN s.PROG_STD_TC > 0                     THEN 'SEASON TO DATE'
                WHEN pk.PRIOR_KEY IS NOT NULL
                 AND s.IS_YEAR_LABEL                       THEN 'PRIOR FISCAL YEAR'
                WHEN pk.PRIOR_KEY IS NOT NULL              THEN 'PRIOR SEASON'
                ELSE 'NO NORM' END                                AS NORM_BASIS
    FROM std s
    LEFT JOIN prior_pick pk
      ON  pk.NETWORK_CODE = s.NETWORK_CODE
      AND pk.PROGRAM_CODE = s.PROGRAM_CODE
      AND pk.NORM_KEY     = s.NORM_KEY
    LEFT JOIN prior_prem pm
      ON  pm.NETWORK_CODE = s.NETWORK_CODE
      AND pm.PROGRAM_CODE = s.PROGRAM_CODE
      AND pm.NORM_KEY     = s.NORM_KEY
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    NETWORK_CODE                 AS NETWORK,
    BROADCAST_DATE               AS AIR_DATE,
    DAY_OF_WEEK                  AS DAY,
    PROGRAM_NAME                 AS PROGRAM,
    EPISODE_NAME                 AS EPISODE,
    CT_SEASON                    AS SEASON,
    CONTENT_TYPE,
    START_TIME,
    BCAST_START_2959             AS START_2959,
    ROUND(DEMO_AA/1000,0)        AS TELECAST_000,
    ROUND(EFF_NORM/1000,0)       AS SEASON_NORM_000,
    EFF_TC                       AS SEASON_NORM_TC,
    NORM_BASIS,
    ROUND((ROUND(DEMO_AA/1000,0)
           / NULLIF(ROUND(EFF_NORM/1000,0),0) - 1) * 100, 1)      AS CHANGE_VS_NORM_PCT,
    IFF(NORM_BASIS IN ('PRIOR SEASON','PRIOR FISCAL YEAR'),
        PRIOR_KEY, NULL)                                          AS PRIOR_SEASON,
    IFF(NORM_BASIS IN ('PRIOR SEASON','PRIOR FISCAL YEAR'),
        ROUND(PRIOR_PREM_AVG/1000,0), NULL)                       AS PRIOR_PREMIERE_000,
    IFF(NORM_BASIS IN ('PRIOR SEASON','PRIOR FISCAL YEAR'),
        PRIOR_PREM_TC, NULL)                                      AS PRIOR_PREMIERE_TC,
    IFF(NORM_BASIS IN ('PRIOR SEASON','PRIOR FISCAL YEAR'),
        ROUND((ROUND(DEMO_AA/1000,0)
               / NULLIF(ROUND(PRIOR_PREM_AVG/1000,0),0) - 1) * 100, 1), NULL)
                                                     AS CHANGE_VS_PRIOR_PREMIERE_PCT,
    STD_RANK                     AS EP_RANK_IN_SEASON,
    STD_UNIVERSE                 AS EPS_IN_SEASON,
    CASE WHEN EFF_TC IS NULL OR EFF_TC = 0
              THEN 'NO NORM: SEASON PREMIERE WITH NO PRIOR SEASON'
         WHEN PROG_STD_TC = 0
              THEN 'SEASON PREMIERE: NORMED ON PRIOR SEASON'
         WHEN EFF_TC < 3
              THEN 'THIN NORM: ' || EFF_TC || ' TELECAST(S)'
    END                          AS NORM_CAVEAT,
    $eff_target             AS TO_DATE
FROM joined
WHERE ( ($p_program IS NULL
         AND BROADCAST_DATE BETWEEN COALESCE(TO_DATE($p_report_from), $eff_target)
                                AND COALESCE(TO_DATE($p_report_to),   $eff_target))
     OR  $p_program IS NOT NULL )
ORDER BY BROADCAST_DATE DESC, DEMO_AA DESC;

## 1.2 Quarter-Hour Movement

**A&E case:** OV_IND_2: quarter-hour breakdown for THE UNXPLAINED on HIST, 6/12/26

**Expected:** QH1 151, QH2 168, QH3 168, QH4 141


In [ ]:
%%sql
SET p_network           = 'HIST';
SET p_rating_src        = 'Live+SD';
SET p_target_date       = '2026-06-12';
SET p_season            = NULL;
SET p_include_leadin_qh = FALSE;
SET p_show_runover      = FALSE;
SET p_runover_max_min   = 15;
SET p_win_start_2959    = 2000;
SET p_win_end_2959      = 2400;
SET p_demo = NULL;
SET eff_demo = (SELECT COALESCE($p_demo, 'DEMO_AA'));
SET eff_target = (SELECT COALESCE(TO_DATE($p_target_date),
                   (SELECT LATEST_QH_BROADCAST_DATE
                    FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                    WHERE NETWORK_CODE = $p_network
                      AND RATING_SOURCE = $p_rating_src)));

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH
target AS (
    SELECT DISTINCT f2.NETWORK_CODE, f2.PROGRAM_CODE, f2.CT_SEASON
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT f2
    WHERE f2.BROADCAST_DATE = $eff_target
      AND f2.IS_FIRST_RUN_CT
      AND ($p_rating_src IS NULL OR f2.RATING_SOURCE = $p_rating_src)
      AND ($p_network    IS NULL OR f2.NETWORK_CODE  = $p_network)
      AND ($p_win_start_2959 IS NULL OR f2.EP_START_2959 >= $p_win_start_2959)
      AND ($p_win_end_2959   IS NULL OR f2.EP_START_2959 <  $p_win_end_2959)
      AND f2.CT_SEASON IS NOT NULL
      AND ($p_season IS NULL OR f2.CT_SEASON = $p_season)
),
prem AS (
    SELECT
        f.NETWORK_CODE, f.PROGRAM_CODE, f.TELECAST_NUM,
        f.PROGRAM_NAME, f.EPISODE_NAME, f.CT_SEASON,
        t.CT_SEASON AS NORM_KEY,
        f.BROADCAST_DATE, f.DAY_OF_WEEK,
        f.QTR_START_TIME, f.QH_START_2959, f.EP_START_2959,
        CAST(f.QH_MIN AS INT) AS QH_MIN,
        f.IS_FIRST_RUN_CT, IDENTIFIER($eff_demo) AS DEMO_AA
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT f
    JOIN target t
      ON  t.NETWORK_CODE = f.NETWORK_CODE
      AND t.PROGRAM_CODE = f.PROGRAM_CODE
      AND f.CT_SEASON = t.CT_SEASON
    WHERE ($p_rating_src IS NULL OR f.RATING_SOURCE = $p_rating_src)
      AND f.BROADCAST_DATE <= $eff_target
      AND f.IS_FIRST_RUN_CT
),
span AS (
    SELECT NETWORK_CODE, PROGRAM_CODE, NORM_KEY,
           MIN(BROADCAST_DATE) AS SEASON_FROM,
           MAX(BROADCAST_DATE) AS SEASON_TO
    FROM prem
    GROUP BY NETWORK_CODE, PROGRAM_CODE, NORM_KEY
),
leadin AS (
    SELECT
        f.NETWORK_CODE, f.PROGRAM_CODE, f.TELECAST_NUM,
        f.PROGRAM_NAME, f.EPISODE_NAME, f.CT_SEASON,
        s.NORM_KEY,
        f.BROADCAST_DATE, f.DAY_OF_WEEK,
        f.QTR_START_TIME, f.QH_START_2959, f.EP_START_2959,
        CAST(f.QH_MIN AS INT) AS QH_MIN,
        f.IS_FIRST_RUN_CT, IDENTIFIER($eff_demo) AS DEMO_AA
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT f
    JOIN span s
      ON  s.NETWORK_CODE = f.NETWORK_CODE
      AND s.PROGRAM_CODE = f.PROGRAM_CODE
    WHERE $p_include_leadin_qh
      AND NOT f.IS_FIRST_RUN_CT
      AND ($p_rating_src IS NULL OR f.RATING_SOURCE = $p_rating_src)
      AND f.BROADCAST_DATE BETWEEN s.SEASON_FROM AND s.SEASON_TO
),
base AS (
    SELECT * FROM prem
    UNION ALL
    SELECT * FROM leadin
),
seq AS (
    SELECT b.*,
        ROW_NUMBER() OVER (PARTITION BY NETWORK_CODE, BROADCAST_DATE,
                           PROGRAM_CODE, TELECAST_NUM
                           ORDER BY QH_START_2959)                    AS QH_NUM,
        COUNT(*)     OVER (PARTITION BY NETWORK_CODE, BROADCAST_DATE,
                           PROGRAM_CODE, TELECAST_NUM)                AS TOTAL_QH
    FROM base b
),
flagged AS (
    SELECT s.*,
        CASE WHEN QH_NUM = TOTAL_QH AND QH_MIN < $p_runover_max_min
             THEN 1 ELSE 0 END                                        AS IS_RUNOVER
    FROM seq s
),
norm AS (
    SELECT NETWORK_CODE, PROGRAM_CODE, NORM_KEY, IS_FIRST_RUN_CT, QH_NUM,
           SUM(DEMO_AA * QH_MIN) / NULLIF(SUM(QH_MIN), 0) AS NORM_AA,
           COUNT(*)     AS NORM_TC
    FROM flagged
    WHERE BROADCAST_DATE <> $eff_target
      AND ($p_show_runover OR IS_RUNOVER = 0)
    GROUP BY NETWORK_CODE, PROGRAM_CODE, NORM_KEY, IS_FIRST_RUN_CT, QH_NUM
),
kept AS (
    SELECT * FROM flagged
    WHERE ($p_show_runover OR IS_RUNOVER = 0)
      AND ( IS_FIRST_RUN_CT
            OR BROADCAST_DATE <> $eff_target
            OR QH_NUM = TOTAL_QH )
),
joined AS (
    SELECT k.*, n.NORM_AA, n.NORM_TC
    FROM kept k
    LEFT JOIN norm n
      ON  n.NETWORK_CODE = k.NETWORK_CODE
      AND n.PROGRAM_CODE = k.PROGRAM_CODE
      AND EQUAL_NULL(n.NORM_KEY, k.NORM_KEY)
      AND n.IS_FIRST_RUN_CT   = k.IS_FIRST_RUN_CT
      AND n.QH_NUM       = k.QH_NUM
),
night AS (
    SELECT * FROM joined WHERE BROADCAST_DATE = $eff_target
),
calc AS (
    SELECT n.*,
        CASE WHEN $p_include_leadin_qh
             THEN LAG(DEMO_AA) OVER (PARTITION BY NETWORK_CODE, BROADCAST_DATE
                                     ORDER BY QH_START_2959)
             ELSE LAG(DEMO_AA) OVER (PARTITION BY NETWORK_CODE, BROADCAST_DATE,
                                     PROGRAM_CODE, TELECAST_NUM
                                     ORDER BY QH_NUM)
        END                                                           AS PREV_AA,
        LAG(NORM_AA) OVER (PARTITION BY NETWORK_CODE, BROADCAST_DATE,
                           PROGRAM_CODE, TELECAST_NUM
                           ORDER BY QH_NUM)                           AS PREV_NORM_AA,
        FIRST_VALUE(DEMO_AA) OVER (PARTITION BY NETWORK_CODE, BROADCAST_DATE,
                           PROGRAM_CODE, TELECAST_NUM ORDER BY QH_NUM
                           ROWS BETWEEN UNBOUNDED PRECEDING
                                    AND UNBOUNDED FOLLOWING)          AS QH1_AA,
        FIRST_VALUE(NORM_AA) OVER (PARTITION BY NETWORK_CODE, BROADCAST_DATE,
                           PROGRAM_CODE, TELECAST_NUM ORDER BY QH_NUM
                           ROWS BETWEEN UNBOUNDED PRECEDING
                                    AND UNBOUNDED FOLLOWING)          AS QH1_NORM_AA,
        MAX(CASE WHEN IS_RUNOVER = 0 THEN QH_NUM END)
            OVER (PARTITION BY NETWORK_CODE, BROADCAST_DATE,
                  PROGRAM_CODE, TELECAST_NUM)                         AS LAST_FULL_QH
    FROM night n
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    NETWORK_CODE     AS NETWORK,
    BROADCAST_DATE   AS AIR_DATE,
    DAY_OF_WEEK      AS DAY,
    PROGRAM_NAME     AS PROGRAM,
    EPISODE_NAME     AS EPISODE,
    CT_SEASON        AS SEASON,
    IFF(NOT IS_FIRST_RUN_CT, 'LEAD-IN (REPEAT)', 'QH' || QH_NUM)           AS QUARTER_HOUR,
    QTR_START_TIME   AS QH_START,
    QH_START_2959    AS QH_START_2959,
    QH_MIN, IS_RUNOVER,
    ROUND(DEMO_AA/1000,0)                                             AS AA_000,
    CASE WHEN PREV_AA IS NOT NULL
         THEN ROUND((ROUND(DEMO_AA/1000,0)
                     / NULLIF(ROUND(PREV_AA/1000,0),0) - 1) * 100, 1)
    END                                                               AS VS_PRIOR_QH_PCT,
    ROUND(NORM_AA/1000,0)                                             AS NORM_AA_000,
    CASE WHEN PREV_NORM_AA IS NOT NULL
         THEN ROUND((ROUND(NORM_AA/1000,0)
                     / NULLIF(ROUND(PREV_NORM_AA/1000,0),0) - 1) * 100, 1)
    END                                                               AS NORM_VS_PRIOR_QH_PCT,
    NORM_TC,
    ROUND((ROUND(MAX(CASE WHEN QH_NUM = LAST_FULL_QH THEN DEMO_AA END)
                 OVER (PARTITION BY NETWORK_CODE, BROADCAST_DATE,
                       PROGRAM_CODE, TELECAST_NUM) / 1000, 0)
           / NULLIF(ROUND(QH1_AA/1000,0),0) - 1) * 100, 1)            AS FINAL_VS_QH1_PCT,
    ROUND((ROUND(MAX(CASE WHEN QH_NUM = LAST_FULL_QH THEN NORM_AA END)
                 OVER (PARTITION BY NETWORK_CODE, BROADCAST_DATE,
                       PROGRAM_CODE, TELECAST_NUM) / 1000, 0)
           / NULLIF(ROUND(QH1_NORM_AA/1000,0),0) - 1) * 100, 1)       AS FINAL_VS_QH1_NORM_PCT,
    $eff_target             AS TO_DATE
FROM calc
ORDER BY EP_START_2959, QH_NUM;

## 1.3 Lifetime Movie Retention Norm


In [ ]:
%%sql
SET p_network      = 'LIF';
SET p_rating_src   = 'Live+SD';
SET p_demo         = 'F25_64_AA_ESTIMATES';
SET eff_demo = (SELECT COALESCE($p_demo, 'DEMO_AA'));
SET p_prem_start   = '20:00:00';
SET p_leadin_start = '18:00:00';
SET p_min_dur      = 15;
SET p_gate_by_day  = TRUE;
SET p_movie_type_col = 'MOVIE_TYPE';
SET win_to_d   = (SELECT LATEST_FULL_SUNDAY FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                  WHERE NETWORK_CODE = $p_network AND RATING_SOURCE = $p_rating_src);
SET win_from_d = (SELECT DATEADD('day', 1, DATEADD('month', -12, $win_to_d)));
SET orig_from  = $win_from_d;  SET orig_to = $win_to_d;
SET acq_from   = $orig_from;   SET acq_to  = $orig_to;

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH base AS (
    SELECT
        PROGRAM_NAME, BROADCAST_DATE, DAY_OF_WEEK,
        IS_FIRST_RUN_CT, START_TIME, END_TIME,
        CONTRIBUTING_DURATION AS CONTRIB_DUR,
        IDENTIFIER($eff_demo)         AS DEMO_AA,
        IDENTIFIER($p_movie_type_col) AS MOVIE_TYPE,
        IS_HOLIDAY
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
    WHERE MOVIE_IND = 1
      AND ($p_network    IS NULL OR NETWORK_CODE  = $p_network)
      AND ($p_rating_src IS NULL OR RATING_SOURCE = $p_rating_src)
),
pairs AS (
    SELECT
        p.MOVIE_TYPE AS PREM_TYPE,
        l.MOVIE_TYPE AS LEADIN_TYPE,
        IFF(NOT l.IS_FIRST_RUN_CT, 'REPEAT', 'PREMIERE') AS LEADIN_AIRING,
        p.IS_HOLIDAY,
        p.DEMO_AA AS PREM_AA, l.DEMO_AA AS LEADIN_AA,
        p.CONTRIB_DUR AS PREM_W, l.CONTRIB_DUR AS LEADIN_W
    FROM base p
    JOIN base l
      ON  l.BROADCAST_DATE = p.BROADCAST_DATE
      AND l.END_TIME       = p.START_TIME
      AND l.MOVIE_TYPE IS NOT NULL
      AND ($p_leadin_start IS NULL OR l.START_TIME  = $p_leadin_start)
      AND ($p_min_dur      IS NULL OR l.CONTRIB_DUR >= $p_min_dur)
    WHERE p.IS_FIRST_RUN_CT
      AND p.MOVIE_TYPE IN ('ORIGINAL','ACQUIRED')
      AND ($p_prem_start IS NULL OR p.START_TIME = $p_prem_start)
      AND (   (p.MOVIE_TYPE = 'ORIGINAL'
                AND ($p_gate_by_day = FALSE OR p.DAY_OF_WEEK = 'Saturday')
                AND p.BROADCAST_DATE >= $orig_from AND p.BROADCAST_DATE <= $orig_to)
           OR (p.MOVIE_TYPE = 'ACQUIRED'
                AND ($p_gate_by_day = FALSE OR p.DAY_OF_WEEK = 'Sunday')
                AND p.BROADCAST_DATE >= $acq_from  AND p.BROADCAST_DATE <= $acq_to) )
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    PREM_TYPE, LEADIN_TYPE, LEADIN_AIRING, IS_HOLIDAY,
    COUNT(*)                                          AS TC,
    ROUND((SUM(LEADIN_AA * LEADIN_W) / NULLIF(SUM(LEADIN_W), 0))/1000,0)                      AS LEADIN_000,
    ROUND((SUM(PREM_AA * PREM_W) / NULLIF(SUM(PREM_W), 0))/1000,0)                        AS PREM_000,
    ROUND(ROUND((SUM(PREM_AA * PREM_W) / NULLIF(SUM(PREM_W), 0))/1000,0)
          / NULLIF(ROUND((SUM(LEADIN_AA * LEADIN_W) / NULLIF(SUM(LEADIN_W), 0))/1000,0),0), 4) AS RETENTION_NORM,
    $win_from_d             AS FROM_DATE,
    $win_to_d               AS TO_DATE
FROM pairs
GROUP BY PREM_TYPE, LEADIN_TYPE, LEADIN_AIRING, IS_HOLIDAY
ORDER BY PREM_TYPE, LEADIN_TYPE, LEADIN_AIRING, IS_HOLIDAY;

## 1.3b Lead-In Retention vs Season Norm

**A&E case:** OV_IND_3: lead-in retention for THE UNXPLAINED on HIST, 6/12/26

**Expected:** lead-in 114 -> programme 158


In [ ]:
%%sql
SET p_network          = 'HIST';
SET p_rating_src       = 'Live+SD';
SET p_target_date      = '2026-06-12';
SET p_season           = NULL;
SET p_min_leadin_dur   = 15;
SET p_include_specials = TRUE;
SET p_win_start_2959   = 2000;
SET p_win_end_2959     = 2400;
SET p_demo = NULL;
SET eff_demo = (SELECT COALESCE($p_demo, 'DEMO_AA'));
SET eff_target = (SELECT COALESCE(TO_DATE($p_target_date),
                   (SELECT MAX(LATEST_BROADCAST_DATE)
                    FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                    WHERE ($p_network IS NULL OR NETWORK_CODE = $p_network)
                      AND RATING_SOURCE = $p_rating_src)));

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH
fact AS (
    SELECT NETWORK_CODE, PROGRAM_CODE, TELECAST_NUM,
           PROGRAM_NAME, EPISODE_NAME, CT_SEASON,
           BROADCAST_DATE, DAY_OF_WEEK, START_TIME, END_TIME, BCAST_START_2959,
           CONTRIBUTING_DURATION, IS_FIRST_RUN_CT, CONTENT_TYPE,
           IDENTIFIER($eff_demo) AS DEMO_AA
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
    WHERE ($p_network    IS NULL OR NETWORK_CODE  = $p_network)
      AND ($p_rating_src IS NULL OR RATING_SOURCE = $p_rating_src)
      AND BROADCAST_DATE <= $eff_target
      AND ($p_include_specials OR CONTENT_TYPE <> 'Special')
),
target AS (
    SELECT DISTINCT f2.NETWORK_CODE, f2.PROGRAM_CODE, f2.CT_SEASON
    FROM fact f2
    WHERE BROADCAST_DATE = $eff_target
      AND IS_FIRST_RUN_CT
      AND ($p_win_start_2959 IS NULL OR BCAST_START_2959 >= $p_win_start_2959)
      AND ($p_win_end_2959   IS NULL OR BCAST_START_2959 <  $p_win_end_2959)
      AND CT_SEASON IS NOT NULL
      AND ($p_season IS NULL OR CT_SEASON = $p_season)
),
pairs AS (
    SELECT
        p.NETWORK_CODE, p.PROGRAM_CODE, p.PROGRAM_NAME, p.EPISODE_NAME,
        p.CT_SEASON,
        t.CT_SEASON AS NORM_KEY,
        p.BROADCAST_DATE, p.DAY_OF_WEEK,
        p.START_TIME, p.DEMO_AA                       AS PREM_AA,
        p.CONTRIBUTING_DURATION                       AS PREM_W,
        l.PROGRAM_CODE                                AS LEADIN_CODE,
        l.PROGRAM_NAME                                AS LEADIN_PROGRAM,
        IFF(NOT l.IS_FIRST_RUN_CT, 'REPEAT', 'PREMIERE')   AS LEADIN_AIRING,
        l.DEMO_AA                                     AS LEADIN_AA,
        l.CONTRIBUTING_DURATION                       AS LEADIN_W
    FROM fact p
    JOIN target t
      ON  t.NETWORK_CODE = p.NETWORK_CODE
      AND t.PROGRAM_CODE = p.PROGRAM_CODE
      AND p.CT_SEASON = t.CT_SEASON
    JOIN fact l
      ON  l.NETWORK_CODE   = p.NETWORK_CODE
      AND l.BROADCAST_DATE = p.BROADCAST_DATE
      AND l.END_TIME       = p.START_TIME
      AND NOT (l.PROGRAM_CODE = p.PROGRAM_CODE AND l.TELECAST_NUM = p.TELECAST_NUM)
      AND ($p_min_leadin_dur IS NULL OR l.CONTRIBUTING_DURATION >= $p_min_leadin_dur)
    WHERE p.IS_FIRST_RUN_CT
),
tonight AS (
    SELECT
        NETWORK_CODE, PROGRAM_CODE, PROGRAM_NAME, EPISODE_NAME, CT_SEASON, NORM_KEY,
        BROADCAST_DATE, DAY_OF_WEEK, START_TIME,
        'TONIGHT'                       AS SCOPE,
        LEADIN_PROGRAM, LEADIN_AIRING,
        1                               AS TC,
        LEADIN_AA, PREM_AA
    FROM pairs
    WHERE BROADCAST_DATE = $eff_target
),
norm_pool AS (
    SELECT * FROM pairs
    WHERE BROADCAST_DATE <> $eff_target
),
norm_all AS (
    SELECT
        NETWORK_CODE, PROGRAM_CODE, MAX(PROGRAM_NAME) AS PROGRAM_NAME, NORM_KEY,
        'SEASON - ALL LEAD-INS'         AS SCOPE,
        'ALL'                           AS LEADIN_PROGRAM,
        'ALL'                           AS LEADIN_AIRING,
        COUNT(*)                        AS TC,
        SUM(LEADIN_AA * LEADIN_W) / NULLIF(SUM(LEADIN_W), 0)                  AS LEADIN_AA,
        SUM(PREM_AA * PREM_W) / NULLIF(SUM(PREM_W), 0)                    AS PREM_AA
    FROM norm_pool
    GROUP BY NETWORK_CODE, PROGRAM_CODE, NORM_KEY
),
norm_split AS (
    SELECT
        NETWORK_CODE, PROGRAM_CODE, MAX(PROGRAM_NAME) AS PROGRAM_NAME, NORM_KEY,
        'SEASON - BY LEAD-IN'           AS SCOPE,
        MAX(LEADIN_PROGRAM)             AS LEADIN_PROGRAM,
        LEADIN_AIRING,
        COUNT(*)                        AS TC,
        SUM(LEADIN_AA * LEADIN_W) / NULLIF(SUM(LEADIN_W), 0)                  AS LEADIN_AA,
        SUM(PREM_AA * PREM_W) / NULLIF(SUM(PREM_W), 0)                    AS PREM_AA
    FROM norm_pool
    GROUP BY NETWORK_CODE, PROGRAM_CODE, NORM_KEY, LEADIN_CODE, LEADIN_AIRING
),
stacked AS (
    SELECT NETWORK_CODE, PROGRAM_CODE, PROGRAM_NAME, EPISODE_NAME, CT_SEASON, NORM_KEY,
           BROADCAST_DATE, DAY_OF_WEEK, START_TIME,
           SCOPE, LEADIN_PROGRAM, LEADIN_AIRING, TC, LEADIN_AA, PREM_AA, 1 AS SORT_ORD
    FROM tonight
    UNION ALL
    SELECT n.NETWORK_CODE, n.PROGRAM_CODE, n.PROGRAM_NAME, NULL, NULL, n.NORM_KEY,
            NULL, NULL, NULL,
           n.SCOPE, n.LEADIN_PROGRAM, n.LEADIN_AIRING, n.TC, n.LEADIN_AA, n.PREM_AA, 2
    FROM norm_all n
    UNION ALL
    SELECT n.NETWORK_CODE, n.PROGRAM_CODE, n.PROGRAM_NAME, NULL, NULL, n.NORM_KEY,
            NULL, NULL, NULL,
           n.SCOPE, n.LEADIN_PROGRAM, n.LEADIN_AIRING, n.TC, n.LEADIN_AA, n.PREM_AA, 3
    FROM norm_split n
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    NETWORK_CODE   AS NETWORK,
    PROGRAM_CODE,
    PROGRAM_NAME   AS PROGRAM,
    EPISODE_NAME   AS EPISODE,
    CT_SEASON      AS SEASON,
    BROADCAST_DATE AS AIR_DATE,
    SCOPE,
    LEADIN_PROGRAM,
    LEADIN_AIRING,
    TC,
    ROUND(LEADIN_AA/1000,0)                            AS LEADIN_000,
    ROUND(PREM_AA/1000,0)                              AS PROGRAM_000,
    ROUND(ROUND(PREM_AA/1000,0)
          / NULLIF(ROUND(LEADIN_AA/1000,0),0), 4)      AS RETENTION,
    $eff_target             AS TO_DATE
FROM stacked
ORDER BY PROGRAM, SORT_ORD, LEADIN_PROGRAM, LEADIN_AIRING;

## 1.4 Network Daypart Rollup

**A&E case:** OV_SUM night: HIST, 6/12/26

**Expected:** no single published figure; PREMIERES_IN_WINDOW must equal 1.1's row count


In [ ]:
%%sql
SET p_network          = 'HIST';
SET p_rating_src       = 'Live+SD';
SET p_target_date      = '2026-06-12';
SET p_include_specials = TRUE;
SET p_demo = NULL;
SET eff_demo = (SELECT COALESCE($p_demo, 'DEMO_AA'));

SET eff_target = (SELECT COALESCE(TO_DATE($p_target_date),
                   (SELECT MAX(LATEST_QH_BROADCAST_DATE)
                    FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                    WHERE ($p_network IS NULL OR NETWORK_CODE = $p_network)
                      AND RATING_SOURCE = $p_rating_src)));

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH qh AS (
    SELECT
        NETWORK_CODE, BROADCAST_DATE, DAY_OF_WEEK,
        PROGRAM_CODE, TELECAST_NUM, IS_FIRST_RUN_CT, CONTENT_TYPE,
        QH_START_2959,
        CAST(QH_MIN AS FLOAT) AS QH_MIN,
        IDENTIFIER($eff_demo) AS DEMO_AA
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_QH_FACT
    WHERE BROADCAST_DATE = $eff_target
      AND ($p_network    IS NULL OR NETWORK_CODE  = $p_network)
      AND ($p_rating_src IS NULL OR RATING_SOURCE = $p_rating_src)
      AND ($p_include_specials OR CONTENT_TYPE <> 'Special')
),
daypart AS (
              SELECT 1 AS ORD, 'TOTAL DAY 6A-6A'   AS DAYPART,  600 AS FROM_2959, 2960 AS TO_2959
    UNION ALL SELECT 2 AS ORD, 'OVERNIGHT 8P-12A'  AS DAYPART, 2000 AS FROM_2959, 2400 AS TO_2959
    UNION ALL SELECT 3 AS ORD, 'PRIME M-SU 8P-11P' AS DAYPART, 2000 AS FROM_2959, 2300 AS TO_2959
),
joined AS (
    SELECT d.ORD, d.DAYPART,
           q.NETWORK_CODE, q.BROADCAST_DATE, q.DAY_OF_WEEK,
           q.PROGRAM_CODE, q.TELECAST_NUM, q.IS_FIRST_RUN_CT,
           q.QH_MIN, q.DEMO_AA
    FROM qh q
    JOIN daypart d
      ON  q.QH_START_2959 >= d.FROM_2959
      AND q.QH_START_2959 <  d.TO_2959
),
rolled AS (
    SELECT
        NETWORK_CODE, BROADCAST_DATE, DAY_OF_WEEK, ORD, DAYPART,
        COUNT(DISTINCT TO_VARCHAR(PROGRAM_CODE)||'-'||TO_VARCHAR(TELECAST_NUM)) AS TELECASTS,
        COUNT(DISTINCT CASE WHEN IS_FIRST_RUN_CT
                       THEN TO_VARCHAR(PROGRAM_CODE)||'-'||TO_VARCHAR(TELECAST_NUM) END) AS PREMIERES,
        COUNT(*)                                        AS QUARTER_HOURS,
        SUM(QH_MIN)                                     AS MINUTES,
        SUM(DEMO_AA * QH_MIN) / NULLIF(SUM(QH_MIN),0)   AS AA_WTD
    FROM joined
    GROUP BY NETWORK_CODE, BROADCAST_DATE, DAY_OF_WEEK, ORD, DAYPART
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    NETWORK_CODE   AS NETWORK,
    BROADCAST_DATE AS AIR_DATE,
    DAY_OF_WEEK    AS DAY,
    DAYPART,
    TELECASTS,
    PREMIERES,
    MAX(CASE WHEN ORD = 2 THEN PREMIERES END)
        OVER (PARTITION BY NETWORK_CODE)            AS PREMIERES_IN_WINDOW,
    QUARTER_HOURS,
    MINUTES,
    ROUND(AA_WTD/1000,0)                            AS AA_000,
    $eff_target             AS TO_DATE
FROM rolled
ORDER BY NETWORK, ORD;

## 1.5 Additional Program Data

**A&E case:** SP_3 programme detail: ALASKA STATE TROOPERS, latest season

**Expected:** season row then one row per episode; season average 425 on P25+, the validated figure


In [ ]:
%%sql
SET p_network          = 'AEN';
SET p_rating_src       = 'Live+SD';
SET p_target_date      = NULL;
SET p_program          = '%ALASKA STATE TROOPERS%';
SET p_season           = NULL;
SET p_demo = 'P25_AA_ESTIMATES';
SET eff_demo = (SELECT COALESCE($p_demo, 'DEMO_AA'));
SET p_include_specials = TRUE;
SET p_win_start_2959   = 2000;
SET p_win_end_2959     = 2400;
SET eff_target = (SELECT COALESCE(TO_DATE($p_target_date),
                   (SELECT MAX(LATEST_BROADCAST_DATE)
                    FROM AUDIENCE_DEV_DB.CRIS.CRIS_LATEST_DATE
                    WHERE ($p_network IS NULL OR NETWORK_CODE = $p_network)
                      AND RATING_SOURCE = $p_rating_src)));

SET demo_used = (SELECT IFF(UPPER(COALESCE($p_demo,'DEMO_AA')) = 'DEMO_AA',
                     CASE $p_network
                          WHEN 'AEN'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'HIST' THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'FYI'  THEN 'P25_64_AA_ESTIMATES'
                          WHEN 'LIF'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'LMN'  THEN 'F25_64_AA_ESTIMATES'
                          WHEN 'VICE' THEN 'P18_49_AA_ESTIMATES'
                          ELSE 'UNRESOLVED - no default demo outside the six'
                     END,
                     UPPER($p_demo)));

WITH
scoped AS (
    SELECT
        NETWORK_CODE, PROGRAM_CODE, TELECAST_NUM,
        PROGRAM_NAME, EPISODE_NAME, CT_SEASON,
        BROADCAST_DATE, DAY_OF_WEEK, START_TIME, BCAST_START_2959,
        CONTRIBUTING_DURATION, CONTENT_TYPE,
        IDENTIFIER($eff_demo) AS DEMO
    FROM AUDIENCE_DEV_DB.CRIS.CRIS_TELECAST_FACT
    WHERE ($p_network    IS NULL OR NETWORK_CODE  = $p_network)
      AND ($p_rating_src IS NULL OR RATING_SOURCE = $p_rating_src)
      AND IS_FIRST_RUN_CT
      AND BROADCAST_DATE <= $eff_target
      AND ($p_include_specials OR CONTENT_TYPE <> 'Special')
),
target AS (
    SELECT f2.NETWORK_CODE, f2.PROGRAM_CODE, f2.CT_SEASON
    FROM scoped f2
    WHERE ( $p_program IS NULL
            AND f2.BROADCAST_DATE = $eff_target
            AND ($p_win_start_2959 IS NULL OR f2.BCAST_START_2959 >= $p_win_start_2959)
            AND ($p_win_end_2959   IS NULL OR f2.BCAST_START_2959 <  $p_win_end_2959) )
       OR ( $p_program IS NOT NULL AND f2.PROGRAM_NAME ILIKE $p_program )
    AND f2.CT_SEASON IS NOT NULL
    AND ($p_season IS NULL OR $p_program IS NULL OR f2.CT_SEASON = $p_season)
    QUALIFY ROW_NUMBER() OVER (PARTITION BY f2.NETWORK_CODE, f2.PROGRAM_CODE
                               ORDER BY f2.BROADCAST_DATE DESC,
                                        f2.TELECAST_NUM DESC) = 1
),
season AS (
    SELECT s.*
    FROM scoped s
    JOIN target t
      ON  t.NETWORK_CODE = s.NETWORK_CODE
      AND t.PROGRAM_CODE = s.PROGRAM_CODE
      AND s.CT_SEASON = t.CT_SEASON
),
rolled AS (
    SELECT
        NETWORK_CODE, PROGRAM_CODE, PROGRAM_NAME,
        CT_SEASON,
        COUNT(*)               AS TCASTS,
        SUM(DEMO * CONTRIBUTING_DURATION) / NULLIF(SUM(CONTRIBUTING_DURATION), 0) AS DEMO_AVG,
        SUM(CONTRIBUTING_DURATION) AS TOTAL_DURATION,
        MIN(BROADCAST_DATE)    AS SEASON_FROM,
        MAX(BROADCAST_DATE)    AS SEASON_TO
    FROM season
    GROUP BY NETWORK_CODE, PROGRAM_CODE, PROGRAM_NAME, CT_SEASON
)
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    NETWORK_CODE          AS NETWORK,
    PROGRAM_NAME          AS PROGRAM,
    CT_SEASON             AS SEASON,
    IFF($p_season IS NOT NULL AND $p_program IS NULL,
        'SEASON PIN IGNORED: needs a programme', NULL) AS SEASON_PIN_IGNORED,
    0                     AS SORT_ORD,
    'SEASON AVERAGE'      AS EPISODE,
    SEASON_FROM           AS AIR_DATE,
    SEASON_TO             AS SEASON_THROUGH,
    NULL                  AS DAY,
    NULL                  AS START_TIME,
    TCASTS,
    ROUND(DEMO_AVG/1000,0) AS AA_000,
    TOTAL_DURATION        AS DURATION,
    $eff_target             AS TO_DATE
FROM rolled
UNION ALL
SELECT
    REPLACE($demo_used, '_AA_ESTIMATES', '') AS DEMO_USED,
    NETWORK_CODE, PROGRAM_NAME, CT_SEASON,
    IFF($p_season IS NOT NULL AND $p_program IS NULL,
        'SEASON PIN IGNORED: needs a programme', NULL),
    1, EPISODE_NAME,
    BROADCAST_DATE, NULL, DAY_OF_WEEK, START_TIME,
    1,
    ROUND(DEMO/1000,0),
    CONTRIBUTING_DURATION,
    $eff_target             AS TO_DATE
FROM season
ORDER BY PROGRAM, SORT_ORD, AIR_DATE;